# 01 — Data Audit: Hospital Readmission Risk
**Project:** Predicting 30-day hospital readmission for diabetic patients
**Data:** UCI Diabetes 130-US Hospitals (1999–2008)

**Goal of this notebook:** download the data, check its quality, define the target, and flag leakage risks before any modelling.

In [ ]:
from pathlib import Path
import io, zipfile
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 60)
RAW = Path("data/raw")
FIG = Path("figures")
RAW.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)

## 1. Download the data (runs once)

In [ ]:
URL = "https://archive.ics.uci.edu/static/public/296/diabetes+130-us+hospitals+for+years+1999-2008.zip"
csv_path = RAW / "diabetic_data.csv"

if not csv_path.exists():
    r = requests.get(URL, timeout=120)
    r.raise_for_status()
    with zipfile.ZipFile(io.BytesIO(r.content)) as z:
        z.extractall(RAW)
    # the zip may nest files in a subfolder: move them up if so
    for f in RAW.rglob("*.csv"):
        if f.parent != RAW:
            f.rename(RAW / f.name)

print(sorted(p.name for p in RAW.glob("*.csv")))

## 2. Load
`?` marks missing values. `keep_default_na=False` matters: pandas would otherwise turn the literal string **"None"** (used in `max_glu_serum` and `A1Cresult` to mean *test not taken*) into NaN.

In [ ]:
df = pd.read_csv(csv_path, na_values=["?"], keep_default_na=False, low_memory=False)
print(df.shape)
df.head()

## 3. Missing values

In [ ]:
missing = (df.isna().mean() * 100).round(1).sort_values(ascending=False)
missing = missing[missing > 0].rename("pct_missing").to_frame()
missing

## 4. Target: 30-day readmission

In [ ]:
print(df["readmitted"].value_counts(normalize=True).round(3))

df["readmit_30"] = (df["readmitted"] == "<30").astype(int)
print(f"\n30-day readmission rate: {df['readmit_30'].mean():.1%}")

## 5. Repeat patients
Patients appear in multiple encounters. If the same patient lands in both train and test sets, the model can memorise them. **Every split later must be grouped by `patient_nbr`.**

In [ ]:
enc_per_patient = df.groupby("patient_nbr").size()
print(f"Encounters: {len(df):,}   Unique patients: {enc_per_patient.size:,}")
print(f"Patients with >1 encounter: {(enc_per_patient > 1).mean():.1%}")
enc_per_patient.describe()

## 6. Leakage check: discharges that make readmission impossible
Patients who died or went to hospice cannot be readmitted, so they would teach the model a false signal.
IDs from `IDS_mapping.csv`: 11, 19, 20, 21 = expired; 13, 14 = hospice.

In [ ]:
EXCLUDE_DISCHARGE = [11, 13, 14, 19, 20, 21]
mask = df["discharge_disposition_id"].isin(EXCLUDE_DISCHARGE)
print(f"Encounters to exclude: {mask.sum():,} ({mask.mean():.1%})")
print(f"Their 30-day readmission rate: {df.loc[mask, 'readmit_30'].mean():.1%}")

## 7. Readmission rate by key factors

In [ ]:
def rate_table(col):
    return (df.groupby(col)["readmit_30"]
              .agg(encounters="size", readmit_rate="mean")
              .assign(readmit_rate=lambda t: (t.readmit_rate * 100).round(1)))

rate_table("age")

In [ ]:
prior = pd.cut(df["number_inpatient"], [-1, 0, 1, 2, 4, np.inf],
               labels=["0", "1", "2", "3-4", "5+"])
tbl = df.groupby(prior, observed=True)["readmit_30"].agg(encounters="size", readmit_rate="mean")
tbl["readmit_rate"] = (tbl["readmit_rate"] * 100).round(1)
tbl

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

age = rate_table("age")
axes[0].bar(age.index, age["readmit_rate"], color="#2a6f97")
axes[0].set_title("30-day readmission rate by age")
axes[0].set_ylabel("% readmitted <30 days")
axes[0].tick_params(axis="x", rotation=45)

axes[1].bar(tbl.index.astype(str), tbl["readmit_rate"], color="#c1666b")
axes[1].set_title("30-day readmission rate by prior inpatient visits")
axes[1].set_xlabel("Inpatient visits in prior year")

plt.tight_layout()
plt.savefig(FIG / "01_readmit_rates.png", dpi=150)
plt.show()

## 8. Other columns to decide on

In [ ]:
for col in ["A1Cresult", "max_glu_serum", "change", "diabetesMed", "admission_type_id"]:
    print(f"--- {col} ---")
    print(rate_table(col), "\n")

In [ ]:
print("Numeric summary")
df.select_dtypes("number").drop(columns=["encounter_id", "patient_nbr"]).describe().T.round(2)

## 9. Findings (fill in after running)
- **Readmission rate:**
- **Columns to drop (mostly missing):**
- **Encounters excluded for leakage:**
- **Strongest early signals:**
- **Implication for splitting:** group by `patient_nbr`